# Week 60, Day 5: Cell types from spike waveforms

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w060/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
fs = 30.0                                           # samples per millisecond
tt = np.arange(60) / fs - 0.5                       # -0.5 to 1.5 ms

def waveform(tp, ratio, width, amp):
    trough = -np.exp(-0.5 * (tt / 0.09) ** 2)
    peak = ratio * np.exp(-0.5 * ((tt - tp) / width) ** 2)
    return amp * (trough + peak)

types = {0: (0.30, 0.65, 0.10), 1: (0.65, 0.35, 0.20), 2: (0.80, 0.18, 0.40)}   # trough-to-peak, ratio, peak width
def units(n, g):
    k = g.integers(0, 3, n)
    W = np.array([waveform(types[c][0] * g.lognormal(0, 0.22), types[c][1] * g.lognormal(0, 0.15),
                           types[c][2] * g.lognormal(0, 0.1), g.uniform(50, 150)) for c in k])
    return W + 4 * g.normal(size=W.shape), k

g0 = np.random.default_rng(605)
Wf, wtype = units(600, g0)
for c, col in zip(range(3), ["C3", "C0", "C2"]):
    plt.plot(tt, (Wf[wtype == c][:15] / -Wf[wtype == c][:15].min(1, keepdims=True)).T, color=col, lw=0.6, alpha=0.5)
plt.xlabel("time (ms)"); plt.ylabel("normalised voltage"); plt.show()

### Exercise: Trough-to-peak and peak-to-trough ratio

Complete features(W), which returns for each waveform the trough-to-peak time in ms (from the minimum to the largest value after it, each located to a fraction of a sample by sub), the peak-to-trough ratio (peak height over trough depth), and the late amplitude 0.8 ms after the trough, relative to the trough depth.

In [ ]:
def sub(w, i):
    """Refine an extremum to a fraction of a sample with a parabola through three points."""
    if 0 < i < len(w) - 1 and (w[i - 1] - 2 * w[i] + w[i + 1]) != 0:
        return i + 0.5 * (w[i - 1] - w[i + 1]) / (w[i - 1] - 2 * w[i] + w[i + 1])
    return float(i)

def features(W):
    out = []
    for w in W:
        i_tr = int(np.argmin(w))
        i_pk = i_tr + int(np.argmax(w[i_tr:]))
        tp = ...
        ratio = ...
        late = w[min(i_tr + 24, len(w) - 1)] / -w[i_tr]     # amplitude 0.8 ms after the trough
        out.append([tp, ratio, late])
    return np.array(out)

F = features(Wf)
for c in range(3):
    print(f"type {c}: mean trough-to-peak {F[wtype == c, 0].mean():.2f} ms, ratio {F[wtype == c, 1].mean():.2f}, late amplitude {F[wtype == c, 2].mean():.2f}")
plt.scatter(F[:, 0], F[:, 1], c=wtype, s=6, cmap="brg"); plt.xlabel("trough-to-peak (ms)"); plt.ylabel("peak / trough"); plt.show()

In [ ]:
import numpy as np
def test_known():
    "features recovers the trough-to-peak time and ratio of a clean waveform"
    f = features(waveform(0.5, 0.4, 0.15, 100.0)[None])[0]
    assert abs(f[0] - 0.5) < 0.05 and abs(f[1] - 0.4) < 0.05
def test_shape():
    "F has one row per unit and three columns"
    assert F.shape == (600, 3)

test_known()
test_shape()
print("All checks passed")

### Exercise: How many types, and which units

Standardise the features and choose the number of mixture components from 1 to 6 by BIC. Then fit 3 components to the features and to the first three principal components of the amplitude-normalised waveforms, and compare each with the true types.

In [ ]:
from itertools import permutations
def mvn_pdf(X, m, S):
    L = np.linalg.cholesky(S); Z = np.linalg.solve(L, (X - m).T)
    return np.exp(-0.5 * (Z ** 2).sum(0) - np.log(np.diag(L)).sum() - 0.5 * len(m) * np.log(2 * np.pi))

def gmm(X, K, g, iters=200, reg=1e-3):
    n, d = X.shape; best = None
    for _ in range(4):
        mu = X[g.choice(n, K, replace=False)]; Sig = np.array([np.cov(X.T)] * K); pi = np.full(K, 1 / K); prev = -np.inf
        for _ in range(iters):
            dens = np.column_stack([pi[k] * mvn_pdf(X, mu[k], Sig[k]) for k in range(K)]) + 1e-300
            ll = np.log(dens.sum(1)).sum()
            if ll - prev < 1e-5: break
            prev = ll; R = dens / dens.sum(1, keepdims=True); Nk = R.sum(0) + 1e-9
            pi, mu = Nk / n, (R.T @ X) / Nk[:, None]
            Sig = np.array([((R[:, k, None] * (X - mu[k])).T @ (X - mu[k])) / Nk[k] + reg * np.eye(d) for k in range(K)])
        if best is None or ll > best[0]: best = (ll, R)
    ll, R = best
    return ll, R.argmax(1), (K - 1) + K * d + K * d * (d + 1) / 2

def agree(lab, truth, k):
    return max(np.mean(np.array(p)[lab] == truth) for p in permutations(range(k)))

gm = np.random.default_rng(6051)
Fz = (F - F.mean(0)) / F.std(0)
bics = []
for K in range(1, 7):
    ll, _, p = gmm(Fz, K, gm)
    bics.append(...)
k_best = int(np.argmin(bics)) + 1
_, lab_feat, _ = gmm(Fz, 3, gm)
Wn = Wf / -Wf.min(1, keepdims=True)
Wc = Wn - Wn.mean(0)
U, S, Vt = np.linalg.svd(Wc, full_matrices=False)
Z3 = ...
Z3 = Z3 / Z3.std(0)
_, lab_pca, _ = gmm(Z3, 3, gm)
acc_feat, acc_pca = agree(lab_feat, wtype, 3), agree(lab_pca, wtype, 3)
print("BIC:", np.round(bics, 0), "-> K =", k_best)
print(f"agreement with true types: features {acc_feat:.3f}, waveform PCs {acc_pca:.3f}")

In [ ]:
import numpy as np
def test_k():
    "BIC on the standardised features chooses at least 3 components"
    assert k_best >= 3
def test_agree():
    "with 3 components, features and waveform principal components both match the types on more than 85% of units"
    assert acc_feat > 0.85 and acc_pca > 0.85

test_k()
test_agree()
print("All checks passed")

### Exercise: How pure is the narrow cluster?

Pick 40 tagged units of type 0, preferring large amplitudes (sample with probability proportional to amplitude). Find the feature-based cluster that holds most of them, the fraction of tagged cells it captures, and a 95% interval for that fraction (Wilson interval).

In [ ]:
amp = -Wf.min(1)
idx0 = np.where(wtype == 0)[0]
gt = np.random.default_rng(6052)
tagged = gt.choice(idx0, 40, replace=False, p=amp[idx0] / amp[idx0].sum())
narrow_cluster = np.bincount(lab_feat[tagged], minlength=3).argmax()
capture = ...
z, n = 1.96, 40
centre = (capture + z ** 2 / (2 * n)) / (1 + z ** 2 / n)
half = z * np.sqrt(capture * (1 - capture) / n + z ** 2 / (4 * n ** 2)) / (1 + z ** 2 / n)
ci_lo, ci_hi = centre - half, centre + half
purity = np.mean(wtype[lab_feat == narrow_cluster] == 0)
print(f"tagged cells in the narrow cluster: {capture:.3f} (95% interval {ci_lo:.2f} to {ci_hi:.2f})")
print(f"purity of the narrow cluster (known only in simulation): {purity:.3f}")

In [ ]:
import numpy as np
def test_capture():
    "the cluster holding most tagged cells contains more than 85% of them"
    assert capture > 0.85
def test_ci():
    "even a perfect capture of 40 cells leaves a 95% interval reaching below 0.95"
    assert ci_lo < 0.95 and ci_hi - ci_lo > 0.05

test_capture()
test_ci()
print("All checks passed")